# U2_T3B · Métodos para raíces complejas: Müller y Bairstow

### Interpolación parabólica y factores cuadráticos

## ¿Qué vas a aprender?

Los métodos cerrados (bisección, falsa posición) y la secante solo encuentran
raíces **reales**: la línea que trazan en el plano real nunca toca una raíz
compleja. Aquí programas dos métodos que sí pueden: **Müller**, que ajusta una
**parábola** por tres puntos, y **Bairstow**, que extrae **factores
cuadráticos** de dos en dos. De paso aplicas lo aprendido en la tarea anterior, **deflactar**, para encontrar todas las raíces de un polinomio.

| # | Pregunta | Tipo | Puntos |
|---|---|---|---|
| 1 | Müller: la parábola por tres puntos | opción múltiple | 10 |
| 2 | Bairstow: factor cuadrático y deflación | opción múltiple | 10 |
| 3 | Programar `muller(f, x0, x1, x2, ...)` | código (casos ocultos) | 25 |
| 4 | La primera aproximación $\omega_3$ | numérica | 10 |
| 5 | Programar `bairstow(a, r0, s0, ...)` | código (casos ocultos) | 25 |
| 6 | La corrección $(\Delta r, \Delta s)$ | 2 casillas | 10 |
| 7 | Comparación con `numpy.roots` | opción múltiple | 10 |

Calificación $= 100\%$ automático: cuentan las 7 preguntas de la tabla.

Las **2 actividades a mano** (secciones 3 y 4) se hacen igual, pero se entregan
como evidencia: una **captura de la hoja de trabajo**, fotos del cuaderno el **enlace de tu
cuaderno en Colab**.

Se requiere **$\ge 90\%$** de los 100 puntos para poder enviar.

In [2]:
#@title ⚙️ Configuración general — ejecuta esta celda **PRIMERO** (no modificar)
# ============================================================
# CELDA 1 — CONFIGURACIÓN GENERAL
# Detecta si corre en Google Colab o en tu computadora, monta Drive y
# carga el motor de la tarea (ver MODO_MOTOR más abajo).
# ============================================================
import base64
import importlib
import os
import subprocess
import sys
import zlib

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4.5)

# ¿Estamos en Google Colab?
try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

# --- Repositorio de la tarea  <-- EDITAR SOLO SI CAMBIA EL REPO ------
REPO_URL = "https://github.com/ITH-MGL-MN/U2_T3B.git"
REPO_NOMBRE = "U2_T3B"

# --- Solo en tu computadora: de dónde sale el motor ---
#   "modulos" -> profe/** tal cual, importado como paquete. Es el modo para
#                DEPURAR: Pylance lo entiende, así que funcionan F12 (ir a
#                definición), el autocompletado, el hover y los puntos de
#                ruptura, y estos caen en el código de verdad.
#   "bundle"  -> grader_local.py, el motor APLANADO en un solo archivo (igual
#                que el blob): para probar la forma exacta que corre en Colab.
#   "blob"    -> grader_ofuscado.txt, lo que recibe el alumno.
# En Colab siempre se usa el blob, haga lo que haga esta variable.
MODO_MOTOR = "modulos"

if ES_COLAB:
    BASE = "/content/drive/MyDrive/CursoMN"
    REPO = os.path.join(BASE, REPO_NOMBRE)
else:
    # En local: buscar la carpeta de la tarea (la que tiene grader_local.py)
    # subiendo de carpeta en carpeta desde la carpeta actual.
    REPO = os.getcwd()
    for _ in range(4):
        if os.path.exists(os.path.join(REPO, "grader_local.py")):
            break
        _padre = os.path.dirname(REPO)
        if _padre == REPO:
            break
        REPO = _padre
    # Respaldo: ruta local de la tarea en tu Drive (ajústala si la moviste)
    if not os.path.exists(os.path.join(REPO, "grader_local.py")):
        REPO = r"o:\Mi unidad\CursoMN\U2_T3B"

MODO = "blob" if ES_COLAB else MODO_MOTOR

print("Modo          :", "Google Colab" if ES_COLAB else "PC local")
print("Carpeta tarea :", REPO)

# ============================================================
# Montar Drive y traer el repositorio  (solo en Colab)
# En tu computadora no hace nada.
# ============================================================
if ES_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(BASE, exist_ok=True)
    try:
        if not os.path.exists(REPO):
            subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
        else:
            subprocess.run(["git", "-C", REPO, "pull"], check=True)
        print("✅ Repositorio listo en Drive.")
    except Exception as _exc:
        print("⚠️ No se pudo clonar/actualizar el repositorio:", _exc)
        print("   Si la carpeta ya existe en tu Drive puedes continuar.")
else:
    print("Modo local: no se monta Drive, uso la carpeta de arriba.")

# ============================================================
# Librerías necesarias
# ============================================================
for _pkg in ("sympy", "numpy", "scipy", "matplotlib", "requests", "ipywidgets"):
    try:
        importlib.import_module(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

# ============================================================
# Motor de la tarea
# ============================================================
# El motor vive aparte del cuaderno y este recibe solo la API que el motor
# declara en su `__all__`. Así el preludio de MATLAB de más abajo no puede
# pisarle los builtins (`max`, `sum`, `abs`, ...); ni el alumno, desde una celda
# de práctica, pisarle un nombre suyo. El motor sigue viendo el cuaderno por la
# pila de llamadas, así que `MI_TAREA` y los widgets `resp_i` viven aquí.
sys.path.insert(0, REPO)

if MODO == "modulos":
    # Import ESTÁTICO a propósito: es lo único que Pylance puede analizar, y el
    # `import *` respeta el `__all__` del motor (si el motor estrena un nombre,
    # el cuaderno lo ve sin tocar nada).
    from profe.ui.cuaderno import *      # noqa: F401,F403
else:
    if MODO == "blob":
        _ruta = os.path.join(REPO, "grader_ofuscado.txt")
        with open(_ruta, encoding="utf-8") as f:
            _blob = f.read().strip()
        codigo = zlib.decompress(base64.b64decode(_blob)).decode("utf-8")
    else:
        _ruta = os.path.join(REPO, "grader_local.py")
        with open(_ruta, encoding="utf-8") as f:
            codigo = f.read()

    _MOTOR = {}
    try:
        exec(codigo, _MOTOR)
    except Exception as _exc:
        raise RuntimeError(
            "No pude cargar el motor desde %s.\n"
            "En Colab: revisa que el repositorio se haya clonado.\n"
            "En local: ejecuta  python tools/build.py  para generarlo.\n"
            "Detalle: %r" % (_ruta, _exc)
        )
    globals().update({_n: _MOTOR[_n] for _n in _MOTOR["__all__"]})

print("Motor cargado :", MODO)

# ============================================================
# Preludio estilo MATLAB (el mismo de las demás tareas)
# ============================================================
# Deja a mano `linspace`, `zeros`, `ones`, `eye`, `pi`, `sin`, `plot`, ... y las
# funciones de NumPy también SIN el prefijo `np.`. `max` y `min` van aparte: el
# preludio los redefine para que funcionen como en MATLAB y como en Python
# (`max(a, b)` con dos escalares o elemento a elemento, `max(seq)` para el
# máximo de una secuencia) — está en lib/matlab_like.py. Ojo: `sum`, `abs`,
# `any`, `all` y `round` SÍ siguen siendo los de NumPy.
sys.path.insert(0, os.path.join(REPO, "lib"))
with open(os.path.join(REPO, "lib", "matlab_like.py"), encoding="utf-8") as f:
    exec(f.read(), globals())

# El preludio trae su propio tamaño de figura; el del cuaderno manda.
plt.rcParams["figure.figsize"] = (9, 4.5)
print("Preludio MATLAB:", os.path.join(REPO, "lib", "matlab_like.py"))

# ============================================================
# Herramientas de visualización (tablas y gráficas de los laboratorios)
# ============================================================
# Viven en tools/visualizar.py, FUERA del motor: no forman parte del sistema de
# calificación, así que no hay nada que esconder. Puedes leerlas, copiarlas y
# usarlas en tus propios cuadernos.
sys.path.append(os.path.join(REPO, "tools"))
try:
    import visualizar
    # OJO: `import` no vuelve a leer el archivo si el módulo ya estaba cargado,
    # así que al editar tools/visualizar.py hay que recargarlo o esta celda
    # seguiría usando la copia vieja que quedó en memoria.
    importlib.reload(visualizar)
    from visualizar import (bits_flotante, graficar_errores, graficar_polinomio,
                            latex_cientifico, tabla_experimento)
except ImportError as _exc:
    raise RuntimeError(
        "No encontré tools/visualizar.py, que los laboratorios usan para\n"
        "imprimir sus tablas y gráficas. En la carpeta de la tarea ese archivo\n"
        "debe estar junto al cuaderno.\nDetalle: %r" % (_exc,)
    )
print("Visualización :", os.path.join(REPO, "tools", "visualizar.py"))

print("\nListo. Ahora ejecuta la celda de identificación.")

Modo          : PC local
Carpeta tarea : o:\Mi unidad\CursoMN\U2_T3B
Modo local: no se monta Drive, uso la carpeta de arriba.
Motor cargado : modulos
Preludio MATLAB: o:\Mi unidad\CursoMN\U2_T3B\lib\matlab_like.py
Visualización : o:\Mi unidad\CursoMN\U2_T3B\tools\visualizar.py

Listo. Ahora ejecuta la celda de identificación.


---
## Identifícate

Al ejecutar la celda de configuración, Google te pidió permiso para acceder a
tu Drive. Con esa **misma cuenta** te identifico automáticamente: no tienes que
escribir nada.

> # ⚠️ INICIA SESIÓN CON TU CORREO INSTITUCIONAL
> - Debe ser un correo que termine en **@….tecnm.mx** (por ejemplo `@hermosillo.tecnm.mx`).
> - Si inicias con una cuenta personal, verás una alerta y **no podrás continuar**.
> - Es importante para identificarte y validar tu tarea.

Si ya ejecutaste esta celda varias veces, puede seguirte preguntando por otros
permisos: puedes continuar sin seleccionarlos. Solo se necesita el permiso de
**leer tu correo**.

In [3]:
#@title 👤 Identificación automática
# ============================================================
# IDENTIFICACIÓN AUTOMÁTICA
# Usa el correo institucional (…@….tecnm.mx) con el que montaste Drive.
# De ahí saco tu número de control y tu nombre (Google lo da en Colab).
# ============================================================
if ES_COLAB:
    import requests
    from google.colab import auth
    import google.auth
    import google.auth.transport.requests

    # Autenticación explícita para asegurar el acceso a la identidad
    try:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        auth_request = google.auth.transport.requests.Request()
        creds.refresh(auth_request)
        info = requests.get(
            "https://www.googleapis.com/drive/v3/about?fields=user",
            headers={"Authorization": "Bearer " + creds.token},
            timeout=20,
        ).json()
        cuenta = info.get("user", {})
        email = cuenta.get("emailAddress", "")
        nombre = cuenta.get("displayName", "")
    except Exception as e:
        print("⚠️ No se pudo obtener tu correo automáticamente.")
        print("Asegúrate de permitir el acceso en la ventana emergente.")
        print("Detalle:", e)
        email = ""
        nombre = ""
else:
    # Modo local (pruebas): escribe un correo manualmente
    email = "m16330887@hermosillo.tecnm.mx"
    # El nombre solo lo entrega Google: en local las cuentas no están
    # autenticadas, así que la hoja de trabajo sale nada más con el NC.
    nombre = ""
    # email = input("Correo institucional (pruebas): ").strip()

# --- Validación del dominio institucional ----------------------------
if email and ("@" in email) and email.lower().endswith(".tecnm.mx"):
    alumno_id = email
    print("✅ Identificado:", email)
    print("   Número de control:", extraer_nc(email))
    print("   Nombre:", nombre if nombre else "(no disponible fuera de Colab)")
elif not email:
    alumno_id = None
    print("⛔ No pude identificar tu correo.")
    print("   Vuelve a ejecutar la celda de configuración y acepta el permiso")
    print("   en la ventana de Google.")
else:
    from IPython.display import HTML

    display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;border-radius:8px;'
        'font-weight:bold">⛔ Este correo no es institucional.<br>'
        'Usa tu cuenta @….tecnm.mx. Si iniciaste con otra cuenta, '
        'cierra sesión o usa una ventana de incógnito.</p>'
    ))
    print("Correo detectado:", email)
    alumno_id = None

if alumno_id is None:
    raise RuntimeError(
        "Detente aquí: necesito tu correo institucional para generar tu tarea.\n"
        "Corrige la identificación y vuelve a ejecutar esta celda."
    )

# La tarea se genera UNA sola vez, aquí: las celdas de pregunta solo llaman a
# `pregunta(n)` sobre este MI_TAREA, así que todas hablan de la misma tarea.
MI_TAREA = generar_tarea(alumno_id, nombre)

print("\nListo. Continúa con la sección 1.")

✅ Identificado: m16330887@hermosillo.tecnm.mx
   Número de control: 16330887
   Nombre: (no disponible fuera de Colab)


Esta tarea tiene **7** preguntas automáticas (100 puntos): esas son las que cuentan. Las **2** actividades a mano se hacen igual, pero se entregan como evidencia (captura de la hoja + enlace de tu cuaderno en Colab).


Listo. Continúa con la sección 1.


---
# 1 · Müller: una parábola por tres puntos

Los métodos **cerrados** y la **secante** trazan una línea entre dos puntos y
buscan dónde cruza el eje $x$. Eso solo sirve cuando la raíz es **real**: una
recta en el plano real nunca toca una raíz compleja.

El **método de Müller** reemplaza la recta por una **parábola** que pasa por
**tres** puntos $(x_0, x_1, x_2)$:

$$f_2(x) = a\,(x - x_2)^2 + b\,(x - x_2) + c$$

Al resolver $f_2(x) = 0$ la raíz **puede salir compleja** (cuando
$b^2 - 4ac < 0$), y el método la sigue iterando en el plano complejo. Por eso
Müller encuentra raíces que los métodos anteriores no pueden.

### Las diferencias y los coeficientes

Con $h_0 = x_1 - x_0$, $h_1 = x_2 - x_1$ y las pendientes incrementales
$\delta_0 = \frac{f(x_1)-f(x_0)}{h_0}$,
$\delta_1 = \frac{f(x_2)-f(x_1)}{h_1}$:

$$c = f(x_2),\qquad a = \frac{\delta_1 - \delta_0}{h_1 + h_0},\qquad b = a\,h_1 + \delta_1$$

y la nueva aproximación es

$$x_3 = x_2 + \frac{-2c}{b \pm \sqrt{b^2 - 4ac}}$$

eligiendo el signo $\pm$ que **maximice la magnitud del denominador** (así se
toma la raíz de la parábola más cercana a $x_2$ y se evita la cancelación por
resta). Luego se desplazan los puntos $x_0 \leftarrow x_1$, $x_1 \leftarrow x_2$,
$x_2 \leftarrow x_3$ y se repite.


In [3]:
#@title 🧪 Laboratorio · un paso de Müller, a la vista
import cmath
from IPython.display import Math, Markdown, display

SEP = r',\ '


def _tex(z, dec=6):
    """Un número real o complejo en LaTeX (sin los $ del modo matemático)."""
    z = complex(z)
    re = '%.*g' % (dec, z.real)
    if abs(z.imag) <= 1e-12 * max(1.0, abs(z.real)):
        return re
    return '%s %+.6g\\,i' % (re, z.imag)


# Un polinomio cualquiera con una raíz real cerca de 4 (NO es el tuyo).
def f(x):
    return x**3 - 13*x - 12        # raíces: -3, -1, 4


x0, x1, x2 = 3.2, 4.8, 4.1
f0, f1, f2 = f(x0), f(x1), f(x2)

display(Markdown('**Un paso de Müller.** Con $f(x) = x^{3} - 13x - 12$ y los '
                 'tres puntos $x_0 = %g$, $x_1 = %g$, $x_2 = %g$:'
                 % (x0, x1, x2)))
display(Math(r'f(x_0) = %s,\qquad f(x_1) = %s,\qquad f(x_2) = %s'
             % (_tex(f0), _tex(f1), _tex(f2))))

h0, h1 = x1 - x0, x2 - x1
d0, d1 = (f1 - f0) / h0, (f2 - f1) / h1
a = (d1 - d0) / (h1 + h0)
b = a * h1 + d1
c = f2

display(Math(r'h_0 = x_1 - x_0 = %s,\qquad h_1 = x_2 - x_1 = %s'
             % (_tex(h0), _tex(h1))))
display(Math(r'\delta_0 = \frac{f(x_1) - f(x_0)}{h_0} = %s,\qquad '
             r'\delta_1 = \frac{f(x_2) - f(x_1)}{h_1} = %s'
             % (_tex(d0), _tex(d1))))
display(Math(r'a = \frac{\delta_1 - \delta_0}{h_1 + h_0} = %s,\qquad '
             r'b = a\,h_1 + \delta_1 = %s,\qquad c = f(x_2) = %s'
             % (_tex(a), _tex(b), _tex(c))))

disc = cmath.sqrt(b*b - 4*a*c)
den = b + disc if abs(b + disc) > abs(b - disc) else b - disc
x3 = x2 - 2*c / den

display(Math(r'\sqrt{b^2 - 4ac} = %s \qquad\Longrightarrow\qquad '
             r'x_3 = x_2 - \frac{2c}{b \pm \sqrt{b^2 - 4ac}} = %s'
             % (_tex(disc), _tex(x3))))
display(Markdown('El signo se elige para que $|b \\pm \\sqrt{b^2-4ac}|$ sea el '
                 'mayor: así $x_3$ es la raíz de la parábola más cercana a $x_2$.'))

raiz, nit = muller_referencia(f, x0, x1, x2, tol=1e-9)
display(Markdown('**Iterando hasta converger:**'))
display(Math(r'x = %s \qquad\text{en } %d \text{ iteraciones,}\qquad |f(x)| = %s'
             % (_tex(raiz), nit, latex_cientifico(abs(f(raiz))))))


**Un paso de Müller.** Con $f(x) = x^{3} - 13x - 12$ y los tres puntos $x_0 = 3.2$, $x_1 = 4.8$, $x_2 = 4.1$:

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

El signo se elige para que $|b \pm \sqrt{b^2-4ac}|$ sea el mayor: así $x_3$ es la raíz de la parábola más cercana a $x_2$.

**Iterando hasta converger:**

<IPython.core.display.Math object>

In [4]:
#@title 🧪 Pregunta 1 · Müller: la parábola por tres puntos
pregunta(1)


#### Pregunta 1: Por qué Müller alcanza raíces complejas

**Valor:** $\frac{10}{100}$ puntos

Los métodos cerrados (bisección, falsa posición) y la secante con aritmética **real** no pueden hallar raíces complejas de un polinomio con coeficientes reales. **¿Cómo lo logra Müller?**

a) Porque la parábola siempre tiene raíces reales

b) Porque aproxima las raíces complejas con números reales muy grandes

c) Porque al resolver la ecuación de la parábola la raíz puede salir compleja (la raíz cuadrada de un discriminante negativo), y el método la sigue iterando en el plano complejo

d) Porque deriva la función y usa el criterio de la segunda derivada

RadioButtons(layout=Layout(width='170px'), options=(('a)', 0), ('b)', 1), ('c)', 2), ('d)', 3)), value=None)

---
# 2 · Bairstow: extraer un factor cuadrático

En lugar de buscar las raíces de una en una, Bairstow extrae del polinomio un
**factor cuadrático** $x^2 - r x - s$, y con él saca **dos raíces a la vez**
(reales o complejas conjugadas). Ajusta $r$ y $s$ con un **Newton de dos
variables**:

1. **Primera división** por $x^2 - r x - s$:
   $b_k = a_k + r\,b_{k-1} + s\,b_{k-2}$ (el residuo son los dos últimos:
   $b_1$ y $b_0$).
2. **Segunda división** sobre los $b_k$:
   $c_k = b_k + r\,c_{k-1} + s\,c_{k-2}$ (dan las derivadas parciales del
   residuo).
3. **Sistema $2\times 2$** (con $c_1, c_2, c_3$ y $b_1, b_0$):

$$\begin{bmatrix} c_2 & c_3 \\ c_1 & c_2 \end{bmatrix}
\begin{bmatrix} \Delta r \\ \Delta s \end{bmatrix} =
\begin{bmatrix} -b_1 \\ -b_0 \end{bmatrix}$$

4. Actualiza $r \leftarrow r + \Delta r$, $s \leftarrow s + \Delta s$ y repite.
5. Al converger, las dos raíces del factor son
   $x_{1,2} = \frac{r \pm \sqrt{r^2 + 4s}}{2}$; **deflacta** (baja dos
   grados) y repite con el polinomio reducido hasta agotar el grado.


In [5]:
#@title 🧪 Laboratorio · Bairstow paso a paso y todas las raíces
from IPython.display import Math, Markdown, display

SEP = r',\ '


def _tex(z, dec=6):
    """Un número real o complejo en LaTeX (sin los $ del modo matemático)."""
    z = complex(z)
    re = '%.*g' % (dec, z.real)
    if abs(z.imag) <= 1e-12 * max(1.0, abs(z.real)):
        return re
    return '%s %+.6g\\,i' % (re, z.imag)


def _vec(v):
    return r'\left[%s\right]' % SEP.join(_tex(x) for x in v)


# Un polinomio cualquiera con raíces reales (NO es el tuyo).
a = [1, 6, 11, 6]              # (x+1)(x+2)(x+3)
r, s = -2.5, -1.5              # factor inicial  x^2 - r x - s

display(Markdown('**Un paso de Bairstow.** Con el polinomio '
                 '$P(x) = x^{3} + 6x^{2} + 11x + 6$ y el factor de prueba '
                 '$x^2 - r x - s$ con $r = %g$, $s = %g$:' % (r, s)))

n = len(a) - 1
b = [0.0] * (n + 1); b[0] = a[0]; b[1] = a[1] + r * b[0]
for k in range(2, n + 1):
    b[k] = a[k] + r * b[k - 1] + s * b[k - 2]

c = [0.0] * (n + 1); c[0] = b[0]; c[1] = b[1] + r * c[0]
for k in range(2, n + 1):
    c[k] = b[k] + r * c[k - 1] + s * c[k - 2]

display(Markdown('**1 · Primera división** '
                 '($b_k = a_k + r\\,b_{k-1} + s\\,b_{k-2}$); el residuo son '
                 '$b_1$ y $b_0$:'))
display(Math(r'b = %s' % _vec(b)))
display(Markdown('**2 · Segunda división** '
                 '($c_k = b_k + r\\,c_{k-1} + s\\,c_{k-2}$):'))
display(Math(r'c = %s' % _vec(c)))

b0, b1 = b[n], b[n - 1]
c1, c2, c3 = c[n - 1], c[n - 2], c[n - 3]
D = c2 * c2 - c1 * c3
dr = (b0 * c3 - b1 * c2) / D
ds = (b1 * c1 - b0 * c2) / D

display(Markdown('**3 · Sistema $2\\times 2$** para la corrección:'))
display(Math(r'\begin{bmatrix} %s & %s \\ %s & %s \end{bmatrix}'
             r'\begin{bmatrix} \Delta r \\ \Delta s \end{bmatrix} = '
             r'\begin{bmatrix} %s \\ %s \end{bmatrix}'
             % (_tex(c2), _tex(c3), _tex(c1), _tex(c2), _tex(-b1), _tex(-b0))))
display(Math(r'D = c_2^2 - c_1 c_3 = %s \qquad\Longrightarrow\qquad '
             r'\Delta r = %s,\qquad \Delta s = %s'
             % (_tex(D), _tex(dr), _tex(ds))))
display(Math(r'r \leftarrow r + \Delta r = %s,\qquad '
             r's \leftarrow s + \Delta s = %s'
             % (_tex(r + dr), _tex(s + ds))))

# Todas las raíces, por Bairstow y por numpy.roots (tabla comparativa).
clave = lambda z: (complex(z).real, complex(z).imag)
raices, nit = bairstow_referencia(a, r, s, tol=1e-9)
raices_b = sorted(raices, key=clave)
raices_n = sorted([complex(v) for v in raices_de(a)], key=clave)
filas = len(raices_n) if len(raices_n) >= len(raices_b) else len(raices_b)

tabla_experimento(
    [('Raíz por Bairstow', 'c'), ('Raíz por `numpy.roots`', 'c')],
    [['$%s$' % _tex(raices_b[i]) if i < len(raices_b) else '',
      '$%s$' % _tex(raices_n[i]) if i < len(raices_n) else '']
     for i in range(filas)],
    titulo='### 📊 Todas las raíces del polinomio (deben coincidir)')
display(Math(r'\text{Bairstow: } %d \text{ iteraciones}' % nit))


**Un paso de Bairstow.** Con el polinomio $P(x) = x^{3} + 6x^{2} + 11x + 6$ y el factor de prueba $x^2 - r x - s$ con $r = -2.5$, $s = -1.5$:

**1 · Primera división** ($b_k = a_k + r\,b_{k-1} + s\,b_{k-2}$); el residuo son $b_1$ y $b_0$:

<IPython.core.display.Math object>

**2 · Segunda división** ($c_k = b_k + r\,c_{k-1} + s\,c_{k-2}$):

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 📊 Todas las raíces del polinomio (deben coincidir)

| Raíz por Bairstow | Raíz por `numpy.roots` |
|:---:|:---:|
| $-3$ | $-3$ |
| $-2$ | $-2$ |
| $-1$ | $-1$ |

<IPython.core.display.Math object>

In [6]:
#@title 🧪 Pregunta 2 · Bairstow: factor cuadrático y deflación
pregunta(2)


#### Pregunta 2: Para qué sirven las dos divisiones sintéticas

**Valor:** $\frac{10}{100}$ puntos

En cada iteración de Bairstow se hacen **dos** divisiones sintéticas: la primera da los $b_k$ (y el residuo) y la segunda, sobre los $b_k$, da los $c_k$. **¿Para qué sirven los $c_k$?**

a) Son las derivadas parciales del residuo respecto a $r$ y $s$: con ellas se arma el sistema $2\times2$ de la corrección $(\Delta r, \Delta s)$

b) Son las raíces del polinomio

c) Son los coeficientes del polinomio deflactado

d) No sirven: son un paso redundante

RadioButtons(layout=Layout(width='170px'), options=(('a)', 0), ('b)', 1), ('c)', 2), ('d)', 3)), value=None)

---
# 3 · Actividad a mano · Müller: la primera aproximación $\omega_3$

Aquí abajo está **tu** función y tus tres puntos iniciales. Vas a calcular, a
mano y con calculadora:

1. Evalúa $f$ en los tres puntos.
2. Calcula $h_0, h_1$ y las pendientes $\delta_0, \delta_1$.
3. Arma la parábola ($a, b, c$) y calcula
   $x_3 = x_2 + \frac{-2c}{b \pm \sqrt{b^2-4ac}}$ eligiendo el signo que dé
   el mayor denominador.

Después reporta $\omega_3$ en la celda que sigue. **No redondees los pasos
intermedios**.


In [7]:
#@title ✏️✅ Müller · actividad a mano, práctica y preguntas
EJ_MU = mano_enunciado('MULLER')


### ✏️ Actividad a mano: Método de Müller

**Vibraciones: frecuencia natural con el método de Müller**

La frecuencia de resonancia de un chasis robótico satisface la función no
lineal

$$f(\omega) = \omega^{3} - 13\omega - 10 = 0$$

donde $\omega$ es la frecuencia angular en rad/s.

Utiliza el **método de Müller** con los tres puntos iniciales
$\omega_0 = 3.5$, $\omega_1 = 4.7$ y $\omega_2 = 4$ para estimar la
**primera aproximación** $\omega_3$: ajusta la parábola que pasa por esos tres
puntos y toma su raíz más cercana a $\omega_2$.

**Datos y arranque:** $\omega_0 = 3.5$  ·  $\omega_1 = 4.7$  ·  $\omega_2 = 4$  ·  $f(\omega) = \omega^{3} - 13\omega - 10$

Reporta en la celda siguiente la lista `[x3]`, con al menos 4 decimales. **Se comprueban todos los valores** para que verifiques tu cálculo.

In [8]:
# ---------------------------------------------------------------------
#  Escribe aqui tu PRIMERA aproximación y vuelve a ejecutar esta celda.
# ---------------------------------------------------------------------
MIS_MU = [0.0]          # <-- [omega_3]

mano_comprueba('MULLER', MIS_MU)


**Tus valores reportados** · Vibraciones: frecuencia natural con el método de Müller

| **valor** | **tu resultado** | **referencia** | **error relativo** | **estado** |
|:---|:---|:---|:---|:---|
| $x_3$ | $0$ | $3.9422728$ | $1.0\times 10^{0}$ | ❌ |

_Sugerencias: arrastra TODOS los decimales de los pasos y no redondees a la mitad del cálculo._

---
### 💻 Pregunta 3 · programa el método de Müller

```python
muller(f, x0, x1, x2, tol=1e-6, max_iter=100)  ->  (raiz, n_iter)
```

* `f` — la función a la que buscas la raíz; `x0, x1, x2` — los tres puntos
  iniciales;
* devuelve la raíz (real o compleja) y el número de iteraciones.


In [9]:
#@title Pregunta 3 · programar muller con casos ocultos
pregunta(3)


#### Pregunta 3: Programa: Método de Müller

**Valor:** $\frac{25}{100}$ puntos

Programa el **método de Müller**: con tres puntos iniciales $(x_0, x_1, x_2)$
ajustas una parábola que pasa por los tres y saltas a la raíz de esa parábola
más cercana a $x_2$; repites hasta converger. Como la raíz puede ser compleja,
usa aritmética compleja (`cmath.sqrt`).

Tu función recibe:

* `f` — la función a la que buscas la raíz (una lambda o `def` que recibe `x`)
* `x0, x1, x2` — los tres puntos iniciales
* `tol` — la tolerancia del criterio de paro (por omisión `1e-6`)
* `max_iter` — el número máximo de iteraciones (por omisión `100`)

y devuelve **una tupla con dos elementos**: `(raiz, n_iter)`.

En cada iteración: `h0 = x1-x0`, `h1 = x2-x1`, `d0 = (f(x1)-f(x0))/h0`,
`d1 = (f(x2)-f(x1))/h1`, `a = (d1-d0)/(h1+h0)`, `b = a*h1 + d1`, `c = f(x2)`,
y `x3 = x2 - 2c/(b ± sqrt(b²-4ac))` eligiendo el signo que haga mayor el
denominador; luego desplaza `x0, x1, x2 = x1, x2, x3`.

Con `muller(lambda x: x**3 - 13*x - 12, 4.0, 5.0, 5.5)` debe devolver una raíz
cercana a `4.0` (el polinomio tiene raíces en $-3$, $-1$ y $4$).

Define la función con **exactamente** la firma `muller(f, x0, x1, x2, tol=1e-6, max_iter=100)` en una celda aparte y ejecútala (no la definas dentro de esta celda).

In [10]:
# --- Práctica: programa el método de Müller --------------------------
import cmath
from IPython.display import Math, Markdown, display


def _tex(z, dec=6):
    '''Un número real o complejo en LaTeX (sin los $ del modo matemático).'''
    z = complex(z)
    re = '%.*g' % (dec, z.real)
    if abs(z.imag) <= 1e-12 * max(1.0, abs(z.real)):
        return re
    return '%s %+.6g\\,i' % (re, z.imag)


def muller(f, x0, x1, x2, tol=1e-6, max_iter=100):
    # 1) h0 = x1-x0 ; h1 = x2-x1
    # 2) d0 = (f(x1)-f(x0))/h0 ; d1 = (f(x2)-f(x1))/h1
    # 3) a = (d1-d0)/(h1+h0) ; b = a*h1 + d1 ; c = f(x2)
    # 4) disc = cmath.sqrt(b*b - 4*a*c) ; elige el signo que haga mayor |b ± disc|
    # 5) x3 = x2 - 2*c/(b ± disc) ; desplaza x0,x1,x2 = x1,x2,x3 y repite
    # 6) detente cuando |f(x3)| < tol (o |x3-x2| < tol) o llegues a max_iter
    # return (raiz, n_iter)
    raise NotImplementedError("Completa la función muller")   # borra esta línea


# --- Prueba con TU ejercicio -----------------------------------------
try:
    raiz, nit = muller(EJ_MU['f'], EJ_MU['x0'], EJ_MU['x1'], EJ_MU['x2'])
    display(Markdown('**Prueba con tu ejercicio** '
                     '(con la $f$ y los puntos de tu enunciado):'))
    display(Math(r'x = %s \qquad\text{en } %d \text{ iteraciones,}\qquad '
                 r'|f(x)| = %s'
                 % (_tex(raiz), nit,
                    latex_cientifico(abs(EJ_MU['f'](raiz))))))
except NotImplementedError as exc:
    display(Markdown('⚠️ Todavía no terminas la función: %s' % exc))
except Exception as exc:
    display(Markdown('❌ Tu función falló: `%r`' % (exc,)))


⚠️ Todavía no terminas la función: Completa la función muller

---
### ✏️ Pregunta 4 · la primera aproximación de tu ejercicio

Ya calculaste $\omega_3$ a mano y programaste el método. Contesta con el número
de **tu** ejercicio (el mismo que reportaste en `MIS_MU`).


In [11]:
pregunta(4)      # la primera aproximación omega_3


#### Pregunta 4: $\omega_3$: Método de Müller

**Valor:** $\frac{10}{100}$ puntos

**Ejercicio:** Vibraciones: frecuencia natural con el método de Müller

Con la **función y los tres puntos de tu ejercicio** (los mismos de la
actividad a mano de esta sección), reporta **solo la primera aproximación**
$\omega_3$ del método de Müller.

Recuerda: ajusta la parábola por $(\omega_0, f(\omega_0))$,
$(\omega_1, f(\omega_1))$ y $(\omega_2, f(\omega_2))$, y toma la raíz de esa
parábola más cercana a $\omega_2$ (elige el signo del denominador que dé la
mayor magnitud).

Reporta un **número**, con al menos 4 decimales y **sin unidades**. Si ya
programaste `muller(f, x0, x1, x2)`, el primer valor que devuelve es la raíz
convergida; aquí se te pide solo $\omega_3$, la **primera** iteración.

FloatText(value=0.0, description='Respuesta:', layout=Layout(width='260px'))

---
# 4 · Bairstow: el factor cuadrático y su corrección

Con el **método de Bairstow** extraes del polinomio un factor $x^2 - r x - s$.
Vas a calcular **una iteración** a mano:

1. **Primera división** por $x^2 - r x - s$ (obtienes $b_k$ y el residuo).
2. **Segunda división** sobre los $b_k$ (obtienes $c_k$).
3. Resuelve el sistema $2\times 2$ para la corrección $(\Delta r, \Delta s)$.

Después reporta $\Delta r$ y $\Delta s$ en la celda que sigue.


In [12]:
#@title ✏️✅ Bairstow · actividad a mano y pregunta
EJ_BA = mano_enunciado('BAIRSTOW')


### ✏️ Actividad a mano: Método de Bairstow

**Sistema LTI: polos del polinomio característico (Bairstow)**

El polinomio característico de la función de transferencia de un actuador
mecatrónico (la variable $x$ es la variable $s$ de Laplace) es

$$P(x) = x^{3} + 4x^{2} + 5x + 2 = 0$$

Aplica el **método de Bairstow** partiendo del factor cuadrático de prueba
$x^2 - r x - s$ con $r_0 = -1.5$ y $s_0 = -0.5$, y reporta la **corrección de
la primera iteración** $(\Delta r,\ \Delta s)$.

**Datos y arranque:** $r_0 = -1.5$  ·  $s_0 = -0.5$  ·  coeficientes `a = [1, 4, 5, 2]`

Reporta en la celda siguiente la lista `[Δr, Δs]`, con al menos 4 decimales. **Se comprueban todos los valores** para que verifiques tu cálculo.

In [13]:
# ---------------------------------------------------------------------
#  Escribe aqui los DOS valores de la corrección, en el orden del
#  enunciado: primero Δr y luego Δs.
# ---------------------------------------------------------------------
MIS_BA = [0.0, 0.0]          # <-- [Δr, Δs]

mano_comprueba('BAIRSTOW', MIS_BA)


**Tus valores reportados** · Sistema LTI: polos del polinomio característico (Bairstow)

| **valor** | **tu resultado** | **referencia** | **error relativo** | **estado** |
|:---|:---|:---|:---|:---|
| $\Delta r$ | $0$ | $-0.5$ | $1.0\times 10^{0}$ | ❌ |
| $\Delta s$ | $0$ | $-0.25$ | $1.0\times 10^{0}$ | ❌ |

_Sugerencias: arrastra TODOS los decimales de los pasos y no redondees a la mitad del cálculo._

---
### 💻 Pregunta 5 · programa el método de Bairstow

```python
bairstow(a, r0, s0, tol=1e-6, max_iter=100)  ->  (lista_de_raices, n_iter)
```

* `a` — la lista de coeficientes de mayor a menor grado; `r0, s0` — el factor
  cuadrático inicial;
* devuelve TODAS las raíces (reales y complejas) y el total de iteraciones.


In [14]:
#@title Pregunta 5 · programar bairstow con casos ocultos
pregunta(5)


#### Pregunta 5: Programa: Método de Bairstow

**Valor:** $\frac{25}{100}$ puntos

Programa el **método de Bairstow**: extrae del polinomio un factor cuadrático
$x^2 - r x - s$ por medio de dos divisiones sintéticas y una corrección de
Newton de dos variables; cuando convergen $r$ y $s$, calcula las dos raíces de
ese factor, **deflacta** el polinomio (baja dos grados) y repite hasta agotar
el grado.

Tu función recibe:

* `a` — la lista de coeficientes **de mayor a menor grado** (el polinomio
  tiene grado $n = \text{len}(a)-1$)
* `r0, s0` — el factor cuadrático inicial $x^2 - r x - s$
* `tol` — la tolerancia del criterio de paro (por omisión `1e-6`)
* `max_iter` — el número máximo de iteraciones por factor (por omisión `100`)

y devuelve **una tupla con dos elementos**: `(lista_de_todas_las_raices, n_iter)`,
donde la lista contiene **todas** las raíces (reales y complejas) y `n_iter`
el número total de iteraciones.

En cada iteración: `b_k = a_k + r·b_{k-1} + s·b_{k-2}` (primera división),
`c_k = b_k + r·c_{k-1} + s·c_{k-2}` (segunda), y con
$c_1{=}c_{n-1},\ c_2{=}c_{n-2},\ c_3{=}c_{n-3}$ y $b_1{=}b_{n-1},\ b_0{=}b_n$
resuelves el sistema $2\times2$ para $(\Delta r, \Delta s)$ y actualizas
$r \leftarrow r + \Delta r$, $s \leftarrow s + \Delta s$.

Con `bairstow([1, 6, 11, 6], -2.5, -1.5)` debe devolver las raíces $-1$, $-2$
y $-3$ (el polinomio es $(x+1)(x+2)(x+3)$).

Define la función con **exactamente** la firma `bairstow(a, r0, s0, tol=1e-6, max_iter=100)` en una celda aparte y ejecútala (no la definas dentro de esta celda).

In [15]:
# --- Práctica: programa el método de Bairstow ------------------------
from IPython.display import Math, Markdown, display


def _tex(z, dec=6):
    '''Un número real o complejo en LaTeX (sin los $ del modo matemático).'''
    z = complex(z)
    re = '%.*g' % (dec, z.real)
    if abs(z.imag) <= 1e-12 * max(1.0, abs(z.real)):
        return re
    return '%s %+.6g\\,i' % (re, z.imag)


def bairstow(a, r0, s0, tol=1e-6, max_iter=100):
    # Mientras quede un polinomio de grado >= 3:
    #   1) itera: dos divisiones sintéticas -> (Δr, Δs) -> actualiza r, s
    #   2) al converger, las raíces del factor son (r ± sqrt(r^2+4s))/2
    #   3) deflacta (baja 2 grados) y repite
    # Al final resuelve lo que queda (lineal o cuadrático) directamente.
    # return (lista_de_todas_las_raices, n_iter)
    raise NotImplementedError("Completa la función bairstow")   # borra esta línea


# --- Prueba con TU ejercicio -----------------------------------------
try:
    clave = lambda z: (complex(z).real, complex(z).imag)
    raices, nit = bairstow(EJ_BA['a'], EJ_BA['r0'], EJ_BA['s0'])
    tuyas = sorted([complex(v) for v in raices], key=clave)
    esperadas = sorted([complex(v) for v in raices_de(EJ_BA['a'])], key=clave)
    filas = max(len(tuyas), len(esperadas))
    tabla_experimento(
        [('Raíz de tu `bairstow`', 'c'), ('Raíz de `numpy.roots`', 'c')],
        [['$%s$' % _tex(tuyas[i]) if i < len(tuyas) else '',
          '$%s$' % _tex(esperadas[i]) if i < len(esperadas) else '']
         for i in range(filas)],
        titulo='### 📊 Prueba con tu ejercicio (deben coincidir)')
    display(Math(r'\text{tu bairstow: } %d \text{ iteraciones}' % nit))
except NotImplementedError as exc:
    display(Markdown('⚠️ Todavía no terminas la función: %s' % exc))
except Exception as exc:
    display(Markdown('❌ Tu función falló: `%r`' % (exc,)))


⚠️ Todavía no terminas la función: Completa la función bairstow

---
### ✏️ Pregunta 6 · la corrección de tu ejercicio

Ya calculaste $(\Delta r, \Delta s)$ a mano. Contesta con los dos números de
**tu** ejercicio (los mismos que reportaste en `MIS_BA`).


In [16]:
pregunta(6)      # la corrección (Δr, Δs)


#### Pregunta 6: Corrección de Bairstow: Método de Bairstow

**Valor:** $\frac{10}{100}$ puntos

**Ejercicio:** Sistema LTI: polos del polinomio característico (Bairstow)

Con el **polinomio característico de tu ejercicio** (el de la actividad a mano
de esta sección) y el factor inicial $r_0, s_0$ que te da el enunciado, calcula
**la corrección de la primera iteración** del método de Bairstow.

Pasos:

1. Primera división sintética por $x^2 - r x - s$: obtienes $b_k$.
2. Segunda división (sobre los $b_k$): obtienes $c_k$.
3. Resuelve el sistema $2\times 2$ para $(\Delta r, \Delta s)$.

Escribe **un número en cada casilla**, en el orden del enunciado: primero
$\Delta r$ y luego $\Delta s$.

Escribe **un número en cada casilla**, en el orden del enunciado:

FloatText(value=0.0, description='Δr =', layout=Layout(width='230px'))

FloatText(value=0.0, description='Δs =', layout=Layout(width='230px'))

---
# 5 · Conclusiones

## Lo que hay que llevarse

1. **Los métodos cerrados y la secante no hallan raíces complejas**: la recta
   que trazan en el plano real nunca cruza una raíz compleja. Müller lo resuelve
   con una **parábola** (cuyas raíces sí pueden ser complejas) y Bairstow con un
   **factor cuadrático**.
2. **Müller es la secante con un punto más**: en lugar de una recta por dos
   puntos, una parábola por tres. La nueva aproximación es
   $x_3 = x_2 - \frac{2c}{b \pm \sqrt{b^2-4ac}}$, con el signo que maximiza el
   denominador.
3. **Bairstow es un Newton de dos variables sobre $(r, s)$**: las dos divisiones
   sintéticas dan el residuo y su Jacobiano, y la corrección sale de un sistema
   $2\times 2$.
4. **Deflactar** (bajar el grado tras cada factor) es lo que permite encontrar
   **todas** las raíces sin volver a caer en las ya encontradas.
5. **`numpy.roots`** hace todo esto por ti y más rápido: programar los métodos
   sirve para **entenderlos** y para cuando no tengas `numpy` (un
   microcontrolador, por ejemplo).

## Autoevaluación rápida

* ¿Por qué la secante no puede acercarse a una raíz compleja?
* ¿Qué pasa en Müller si $b^2 - 4ac < 0$?
* ¿Qué son $c_1, c_2, c_3$ en el sistema de Bairstow?
* Si al deflactar el residuo no es cero, ¿qué significa?
* ¿Cuándo conviene tu propio método frente a `numpy.roots`?

---

### ✏️ Pregunta 7


In [17]:
#@title 🧪 Pregunta 7 · comparación con numpy.roots
pregunta(7)


#### Pregunta 7: Raíces múltiples y convergencia

**Valor:** $\frac{10}{100}$ puntos

Un polinomio tiene una raíz **doble** (multiplicidad 2). **¿Qué le pasa a la convergencia de los métodos iterativos (Müller/Bairstow)?**

a) No les afecta en absoluto

b) Convergen en una sola iteración

c) No pueden encontrar raíces dobles

d) Convergen más lento (la raíz es un punto de tangencia, no un cruce); la deflación repetida ayuda a separarla

RadioButtons(layout=Layout(width='170px'), options=(('a)', 0), ('b)', 1), ('c)', 2), ('d)', 3)), value=None)

---
# 6 · Entrega

## Antes de enviar

1. **Revisa tus 7 preguntas.** Si alguna está en blanco el sistema la cuenta
   como error.
2. **Completa `muller` y `bairstow`.** Se prueban con polinomios que nunca ves.
3. Necesitas **$\ge 90\%$** de los 100 puntos para poder enviar.
4. **Haz las 2 actividades a mano** (secciones 3 y 4) y entrega la evidencia:
   una **captura de la hoja de trabajo** (celda de abajo) y el **enlace de tu
   cuaderno en Colab**. La nota sale sola de las preguntas automáticas, pero el
   procedimiento a mano es parte de la tarea y se revisa.

| Sección | Actividad a mano | ¿La reportaste? |
|---|---|---|
| 3 | Müller: la primera aproximación $\omega_3$ | `MIS_MU` |
| 4 | Bairstow: la corrección $(\Delta r, \Delta s)$ | `MIS_BA` |

---

## Las tres celdas del final

| Celda | Qué hace | Cuántas veces |
|---|---|---|
| 📝 **Calificar** | revisa tus respuestas y te dice cuántos puntos llevas. **No envía nada.** | las que quieras |
| 📤 **Enviar** | manda tu calificación a la hoja del profesor. | **solo 2 intentos** |
| 🔎 **Comprobar** | lee lo que la hoja ya tiene guardado de ti. **No envía nada.** | las que quieras |

La **nota** es $100\%$ automático: la de las 7 preguntas. **Calificar
no gasta intentos**: corrige y vuelve a calificar las veces que necesites antes de
enviar.

In [ ]:
#@title 📋 Hoja de trabajo (captura + enlace de Colab)
hoja_manual()

# 📋 Hoja de trabajo manual · U2_T3B

## Método de Müller · Vibraciones: frecuencia natural con el método de Müller

**Datos y arranque:** $\omega_0 = 3.5$  ·  $\omega_1 = 4.7$  ·  $\omega_2 = 4$  ·  $f(\omega) = \omega^{3} - 13\omega - 10$

**Se reporta:** $[x_3]$

| **valor** | **tu resultado** | **error relativo** | **estado** |
|:---|:---|:---|:---|
| $x_3$ | $0$ | $1.0\times 10^{0}$ | ❌ |

...

| **valor** | **tu resultado** | **error relativo** | **estado** |
|:---|:---|:---|:---|
| $\Delta r$ | $0$ | $1.0\times 10^{0}$ | ❌ |
| $\Delta s$ | $0$ | $1.0\times 10^{0}$ | ❌ |

❌ **Revisar**: el error relativo tiene que quedar por debajo de $1.0\times 10^{-3}$.

---

Califica cada rúbrica de 0 a 10 en la hoja U2_T3B_Manual.

Nota final $= 0.70\,(\text{automático}) + 0.30\,(\text{manual})$

In [ ]:
#@title 📝 Calificar (sin enviar) — úsala las veces que quieras
puntos, maximo = calificar()


In [ ]:
#@title 📤 Enviar la calificación (máximo 2 intentos)
enviar(alumno_id)


In [ ]:
#@title 🔎 Comprobar lo que ya está guardado (no envía nada)
consultar_calificacion()
